# Practice Test 2: Advanced Interpretability

**120-minute timer. Same rules.**

**Structure:**
- Q1: Jacobian + gradient through attention
- Q2: Level 4 Boss - O(T) Linear Attention
- Q3: Circuit analysis + SVD — 3 parts
- Q4: Advanced interpretability — 4 parts, **final part intentionally very hard**


In [ ]:
import numpy as np

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def check(name, got, expected=None, cond=None, rtol=1e-4, atol=1e-6):
    try:
        if cond is not None:
            passed = bool(cond(got))
        elif expected is not None:
            passed = np.allclose(np.array(got, dtype=float),
                                 np.array(expected, dtype=float), rtol=rtol, atol=atol)
        else:
            passed = True
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed and expected is not None:
            g, e = np.array(got), np.array(expected)
            print(f"         got shape {g.shape}, expected shape {e.shape}")
            if g.shape == e.shape and g.size <= 16:
                print(f"         got:      {g.ravel()}")
                print(f"         expected: {e.ravel()}")
        return passed
    except Exception as exc:
        print(f"  [ERROR] {name}: {type(exc).__name__}: {exc}")
        return False

print("Helper loaded. softmax and check() are available.")


## Q1: Softmax Jacobian + Gradient through Attention

In [ ]:
def softmax_jacobian(z):
    """
    Goal: Compute the Jacobian matrix of the softmax function. 
    J[i, j] = pi_i * (delta_ij - pi_j), where pi is the softmax output.
    
    Args:
        z: (n,) 1D array of raw logits.
        
    Returns:
        (n, n) Jacobian matrix.
    """
    raise NotImplementedError()

def attention_score_gradient(Q, K, V, dL_dout):
    """
    Goal: Compute the gradient of a scalar loss L w.r.t. the pre-softmax attention 
    scores S. (S = Q @ K.T / sqrt(dk)).
    
    Args:
        Q: (T, dk) Queries.
        K: (T, dk) Keys.
        V: (T, dv) Values.
        dL_dout: (T, dv) The gradient of the loss with respect to the attention output.
        
    Returns:
        (T, T) matrix representing dL/dS.
    """
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ1 TESTS\n" + "="*50)
z = np.array([0.5, -0.3, 1.2, 0.])
pi = softmax(z)
J = softmax_jacobian(z)
check("shape (4,4)", cond=lambda x: x.shape == (4,4), got=J)
check("diagonal: pi*(1-pi)", J.diagonal(), pi*(1-pi))
T, dk, dv = 4, 3, 5
np.random.seed(0)
Q = np.random.randn(T,dk); K = np.random.randn(T,dk)
V = np.random.randn(T,dv); dL = np.random.randn(T,dv)
dS = attention_score_gradient(Q, K, V, dL)
check("gradient shape (T,T)", cond=lambda x: x.shape == (T,T), got=dS)


## Q2: Level 4 Boss - O(T) Linear Attention

In [ ]:
def linear_attention_efficient(Q, K, V):
    """
    Goal: Compute the attention output in O(T) memory and time. Standard attention 
    materializes a (T, T) matrix, which OOMs for large T. Leverage matrix associativity 
    to bypass this completely. Assume no softmax is applied.
    
    Args:
        Q: (T, dk) Queries.
        K: (T, dk) Keys.
        V: (T, dv) Values.
        
    Returns:
        (T, dv) attention output matrix.
    """
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ2 TESTS\n" + "="*50)
T_large, dk_l, dv_l = 1000, 32, 64
np.random.seed(1)
Q_l = np.random.randn(T_large, dk_l); K_l = np.random.randn(T_large, dk_l)
V_l = np.random.randn(T_large, dv_l)
expected_out = (Q_l @ K_l.T) @ V_l
out = linear_attention_efficient(Q_l, K_l, V_l)
check("shape (T, dv)", cond=lambda x: x.shape == (T_large, dv_l), got=out)
check("matches naive (Q @ K.T) @ V", out, expected_out)


## Q3: Circuit Analysis + SVD (3 Parts)

In [ ]:
# Part 1: Compute circuit matrices
def all_circuit_matrices(W_Qs, W_Ks, W_Vs, W_Os):
    """
    Goal: Combine the projection matrices into full QK and OV circuits for every head.
    
    Args:
        W_Qs: (H, d, dk)
        W_Ks: (H, d, dk)
        W_Vs: (H, d, dv)
        W_Os: (H, dv, d)
        
    Returns:
        W_QKs: (H, d, d) The Query-Key circuits.
        W_OVs: (H, d, d) The Output-Value circuits.
    """
    raise NotImplementedError()

# Part 2: SVD
def ov_svd_analysis(W_OVs, threshold=0.9):
    """
    Goal: Perform Singular Value Decomposition on the OV matrices to find their 
    effective rank (the number of singular values needed to capture `threshold` 
    fraction of the Frobenius norm squared).
    
    Args:
        W_OVs: (H, d, d) The Output-Value circuits.
        threshold: float, default 0.9. 
        
    Returns:
        all_sv: (H, d) Singular values, sorted descending.
        eff_ranks: (H,) 1D array of integer effective ranks for each head.
    """
    raise NotImplementedError()

# Part 3: Specificity
def rank_by_specificity(W_OVs, threshold=0.9):
    """
    Goal: Rank heads by how "specific" their OV circuits are. A head is more specific 
    if it has a lower effective rank (i.e., it maps to a tighter subspace).
    
    Args:
        W_OVs: (H, d, d)
        threshold: float, default 0.9.
        
    Returns:
        List of H integers representing the head indices, sorted from most specific 
        (lowest effective rank) to least specific.
    """
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ3 TESTS\n" + "="*50)
H, d_test, dk, dv = 4, 8, 3, 3
W_Qs = np.random.randn(H,d_test,dk); W_Ks = np.random.randn(H,d_test,dk)
W_Vs = np.random.randn(H,d_test,dv); W_Os = np.random.randn(H,dv,d_test)
W_QKs, W_OVs = all_circuit_matrices(W_Qs, W_Ks, W_Vs, W_Os)
check("W_QKs shape (H,d,d)", cond=lambda x: x.shape == (H,d_test,d_test), got=W_QKs)
all_sv, eff_ranks = ov_svd_analysis(W_OVs)
check("sv descending per head", cond=lambda x: np.all(x[:,:-1] >= x[:,1:]), got=all_sv)
ranking = rank_by_specificity(W_OVs)
check("all heads ranked", cond=lambda r: len(r) == H, got=ranking)


## Q4: Advanced Interpretability (4 Parts)

**Complete Parts 1–3 before attempting Part 4.**


In [ ]:
# Part 1: Entropy
def head_entropy_matrix(logits_per_layer):
    """
    Goal: Calculate the average Shannon entropy of the post-softmax attention 
    distribution to identify uniform (uninformative) heads.
    
    Args:
        logits_per_layer: list of n_layers, each containing H arrays of shape (T, T).
        
    Returns:
        (n_layers, H) array of mean entropies.
    """
    raise NotImplementedError()

# Part 2: Induction
def score_induction_heads(weights_per_layer, n):
    """
    Goal: Score heads on how perfectly they match a classic induction head pattern.
    
    Args:
        weights_per_layer: list of n_layers, each containing H arrays of shape (2n, 2n).
        n: Sequence offset to test for induction.
        
    Returns:
        (n_layers, H) array of induction scores in [0, 1].
    """
    raise NotImplementedError()

# Part 3: SAE
def sae_top_k(x, W_enc, b_enc, W_dec, b_dec, k=5):
    """
    Goal: Pass residual stream activations through a Sparse Autoencoder and retrieve 
    the top-k activating features.
    
    Args:
        x: (T, d) Activations.
        W_enc: (d, dh) Encoder weights.
        b_enc: (dh,) Encoder bias.
        W_dec: (dh, d) Decoder weights.
        b_dec: (d,) Decoder bias.
        k: integer. Number of top features to return.
        
    Returns:
        recon: (T, d) Reconstructed activations.
        idx: (T, k) Integer indices of the top-k features.
        vals: (T, k) Float values of the top-k features.
        mse: float representing the Mean Squared Error of the reconstruction.
    """
    raise NotImplementedError()

# Part 4 (HARD): Self-reinforcing circuits
def find_self_reinforcing(W_OVs_L1, W_OVs_L2, W_enc, W_dec, top_k=3):
    """
    Goal: Find SAE features that are heavily self-reinforced by a virtual attention head 
    spanning two layers. A feature self-reinforces if the virtual OV matrix maps its 
    decoder direction back onto its encoder direction.
    
    Args:
        W_OVs_L1: (H1, d, d) Layer 1 OV circuits.
        W_OVs_L2: (H2, d, d) Layer 2 OV circuits.
        W_enc: (d, dh) SAE Encoder.
        W_dec: (dh, d) SAE Decoder.
        top_k: int. Number of tuples to return.
        
    Returns:
        List of top_k tuples in the format (h1, h2, feature_idx, alignment_score),
        sorted descending by the alignment_score.
    """
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ4 PART 4 TESTS (HARD)\n" + "="*50)
H1, H2, d_val, dh_val = 3, 3, 8, 32; np.random.seed(42)
W_OVs_L1 = np.random.randn(H1,d_val,d_val)*0.3
W_OVs_L2 = np.random.randn(H2,d_val,d_val)*0.3
W_enc_s = np.random.randn(d_val,dh_val); W_dec_s = np.random.randn(dh_val,d_val)
feat_dir = np.random.randn(d_val); feat_dir /= np.linalg.norm(feat_dir)
W_enc_s[:,5] = feat_dir*10; W_dec_s[5,:] = feat_dir
W_OVs_L1[0] = np.eye(d_val)*2; W_OVs_L2[1] = np.eye(d_val)*2.5
results = find_self_reinforcing(W_OVs_L1, W_OVs_L2, W_enc_s, W_dec_s, top_k=3)
h1, h2, feat, score = results[0]
check("planted h1=0 is top", h1, 0)
check("planted h2=1 is top", h2, 1)
check("planted feature=5 is top", feat, 5)
check("sorted descending", cond=lambda r: r[0][3] >= r[1][3] >= r[2][3], got=results)
